# Per-Artifact Batch Execute

## What you'll learn

- How the framework decouples overhead amortization from compute dispatch
- What per-artifact dispatch means and why it matters for GPU parallelism
- How to opt out with `per_artifact_dispatch = False` for subprocess-based tools
- The performance model for local, Modal, and opted-out execution

**Prerequisites:** [First Pipeline](../getting-started/01-first-pipeline.ipynb),
[Batching and Performance](03-batching-and-performance.ipynb),
[Compute Routing](13-compute-routing.ipynb).
**Estimated time:** 10 minutes

In [ ]:
from __future__ import annotations

from typing import Any, ClassVar

from artisan.operations.examples import DataGenerator, DataTransformer
from artisan.orchestration import PipelineManager
from artisan.schemas.execution.curator_result import ArtifactResult
from artisan.schemas.specs.input_models import ExecuteInput, PostprocessInput, PreprocessInput
from artisan.utils import tutorial_setup
from artisan.visualization import inspect_pipeline

In [ ]:
env = tutorial_setup("batch_execute")

## The decoupled boundary

`artifacts_per_unit` controls how many artifacts share setup overhead
(Delta scans, sandbox creation, staging writes). With 100 artifacts per
unit, the framework runs 1 Delta scan and writes 5 parquet files for all
100 — not 100 separate scans and 500 files.

But within each unit, **execute is dispatched per-artifact**. On Modal,
this means 100 containers run in parallel instead of processing all 100
sequentially on one container.

```
Unit (100 artifacts):

  prep (batched)                          1 Delta scan, 1 sandbox
    ├─ instantiate_inputs()               All 100 IDs in one query
    ├─ materialize_inputs()               100 files into shared dir
    └─ operation.preprocess()             1 call, returns 100-item lists

  execute (per-artifact)                  100 parallel containers
    ├─ artifact 0  ──→  Modal container
    ├─ artifact 1  ──→  Modal container
    └─ ...         ──→  Modal container

  post (batched)                          1 postprocess, 5 parquet files
    ├─ reassemble results
    ├─ operation.postprocess()            1 call, same data shapes
    └─ record_execution_success()         1 set of staging files
```

| Concern | Parameter | What it controls |
|---------|-----------|------------------|
| Overhead amortization | `artifacts_per_unit` | Delta scans, sandbox, staging |
| Compute parallelism | Per-artifact dispatch | Containers per unit on Modal |

These two concerns are independent. You can have large units (100
artifacts) for overhead amortization while each artifact still gets its
own GPU container.

## Default behavior: per-artifact dispatch

Per-artifact dispatch is the default for all operations. Existing
operations work without changes — the framework handles splitting
preprocess output into per-artifact inputs and reassembling the results
for postprocess.

On the local backend, the per-artifact inputs execute sequentially
(same behavior as before). On Modal, they fan out to parallel containers.

In [ ]:
pipeline = PipelineManager.create(
    name="batch_execute_tutorial",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = pipeline.output

pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 4, "seed": 42},
)
print(f"DataGenerator.per_artifact_dispatch = {DataGenerator.per_artifact_dispatch}")

In [ ]:
pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
    params={"scale_factor": 0.5, "variants": 1},
)

summary = pipeline.finalize()
print(
    f"Pipeline complete: {summary['total_steps']} steps, "
    f"success={summary['overall_success']}"
)
inspect_pipeline(env.delta_root)

## Opting out: `per_artifact_dispatch = False`

Some operations run external tools via `run_command()` where a single
subprocess loads model weights, processes all artifacts, and exits. Each
invocation pays the model-loading cost, so batching amortizes it: 200
artifacts in one subprocess = one model load.

For these operations, set `per_artifact_dispatch = False`. The framework
sends all artifacts to execute in a single call — the same behavior as
before per-artifact dispatch was introduced.

```python
class MyGPUTool(OperationDefinition):
    per_artifact_dispatch: ClassVar[bool] = False
    # ...
```

This is a stopgap. The long-term solution is a persistent executor
pattern where model weights load once and persist across invocations.

## Performance model

At 10,000 units of 100 artifacts each (1M total):

| Metric | Per-artifact units (1M units of 1) | Batched units (10K units of 100) |
|--------|-----------------------------------|----------------------------------|
| Delta scans | 2M | 20K |
| Staging parquet files | 5M | 50K |
| Modal API calls | 1M `remote()` | 10K `spawn_map()` |
| GPU parallelism per unit | 1 container | 100 containers |

The local backend runs the per-artifact loop sequentially — the
parallelism benefit is specific to Modal (and future remote backends).
On the local backend, per-artifact dispatch has no performance impact.

## Summary

| Concept | What it does |
|---------|-------------|
| `artifacts_per_unit` | Controls overhead amortization (Delta scans, staging) |
| Per-artifact dispatch | Fans out execute() per artifact within each unit |
| `per_artifact_dispatch = False` | Keeps all artifacts in one execute call (for subprocess tools) |
| Local backend | Per-artifact loop runs sequentially (no behavior change) |
| Modal backend | Per-artifact inputs dispatch to parallel containers |

Operations work without changes. The framework splits preprocess output,
dispatches per-artifact, and reassembles results for postprocess.

## Next steps

- [Batching and Performance](03-batching-and-performance.ipynb) — `artifacts_per_unit` and overhead amortization
- [Compute Routing](13-compute-routing.ipynb) — Switching between local and Modal compute
- [Running on Modal](14-modal-execution.ipynb) — GPU selection, sandbox transport, debugging
- [Execution Flow](../../concepts/execution-flow.md) — How the framework dispatches and tracks work